# Building Data AI Agent — Production Engineering v2

A portfolio walkthrough of a **bounded LangGraph Text-to-SQL agent**.

`question → route → live schema → generate SQL → AST validation → read-only DB tool → bounded repair → synthesis → trace → diagnosis → evaluation`

The LLM is one component inside a controlled execution harness.


## Architecture and safety contract

```text
User Question
   ↓
Semantic Preflight
   ├─ WRITE / mismatch / ambiguous → SAFE_POLICY_CONTAINMENT
   ↓ READ_QUERY
SQL Generation
   ↓
sqlglot AST + Allowlist Guardrails
   ↓
Read-only PostgreSQL
   ├─ validation/execution failure → at most ONE repair
   ↓
Database-grounded synthesis
   ↓
Trace + Provenance + Diagnosis + Evaluation
```

Implemented boundaries include a one-retry repair cap, read-only database permissions, a **5-second database statement timeout**, and a **200-row result cap**.


## Secret-safe configuration

Publishable notebooks never contain credentials.


In [ ]:
import os
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("DATABASE_URL configured:", bool(os.getenv("DATABASE_URL")))


## Explicit control model

The database statement timeout is a **tool boundary**. The broader 25-second value below is an operational latency threshold used for diagnosis; it is not presented as the database timeout.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ControlPolicy:
    max_sql_retries: int = 1
    max_tool_failures: int = 1
    statement_timeout_ms: int = 5000
    max_result_rows: int = 200
    diagnostic_latency_budget_ms: int = 25000

def diagnose_run(result, policy=ControlPolicy()):
    request_type = result.get("request_type", "UNKNOWN")
    validation = result.get("validation_status")
    execution = result.get("execution_status", "NOT_EXECUTED")
    retries = int(result.get("retry_count", 0) or 0)
    latency = float(result.get("latency_ms", 0.0) or 0.0)

    if latency > policy.diagnostic_latency_budget_ms:
        return {"outcome": "LATENCY_BUDGET_EXCEEDED", "component": "LATENCY"}

    if request_type in {"WRITE_REQUEST", "SCHEMA_MISMATCH", "AMBIGUOUS"}:
        if execution == "NOT_EXECUTED":
            return {"outcome": "SAFE_POLICY_CONTAINMENT", "component": None}
        return {"outcome": "POLICY_CONTAINMENT_FAILURE", "component": "ROUTING_OR_POLICY"}

    if validation == "BLOCKED":
        return {
            "outcome": "REPAIR_EXHAUSTED" if retries else "SQL_VALIDATION_BLOCK",
            "component": "SQL_VALIDATION",
        }

    if execution == "ERROR":
        return {
            "outcome": "REPAIR_EXHAUSTED" if retries else "TOOL_EXECUTION_FAILURE",
            "component": "TOOL_EXECUTION",
        }

    if request_type == "READ_QUERY" and execution == "SUCCESS":
        return {"outcome": "SUCCESS", "component": None}

    return {"outcome": "UNKNOWN_OUTCOME", "component": "UNKNOWN"}


In [ ]:
demo_runs = [
    {"request_type":"READ_QUERY","validation_status":"VALID","execution_status":"SUCCESS","retry_count":0,"latency_ms":850},
    {"request_type":"WRITE_REQUEST","execution_status":"NOT_EXECUTED","retry_count":0,"latency_ms":2},
    {"request_type":"READ_QUERY","validation_status":"VALID","execution_status":"ERROR","retry_count":1,"latency_ms":1400},
]
[diagnose_run(run) for run in demo_runs]


## Evaluation and operational metrics

A serious agent should be evaluated at multiple layers rather than with one aggregate score:

| Layer | Check |
|---|---|
| Routing | Correct class and safe containment |
| SQL safety | AST, table/column allowlists, write rejection |
| Execution | Read-only query completes within bounded tool runtime |
| Semantics | Execution-equivalent result / answer contract |
| Repair | Recovery stays within one retry |
| Observability | Request ID, trace, latency, retries, status |
| Diagnosis | Failure assigned to an actionable component |
| Provenance | Schema/SQL/result/ledger fingerprints |
| Regression | Previously passing cases remain passing |

Useful operational metrics include success rate, safe-containment rate, repair rate, validation-block rate, p50/p95 latency, truncation rate, trace depth, and failure-category distribution.

### Why this is an AI Agent

A prompt wrapper is `prompt → model → answer`.

This system is `question → route → inspect live environment → generate action → validate → call tool → observe → bounded repair → synthesize → trace → diagnose → evaluate`.

**Portfolio framing:** Production-oriented, bounded LangGraph data agent with read-only tool execution, SQL safety guardrails, structured tracing, component-level failure diagnosis, auditable provenance, and regression evaluation.
